# SimCLR Confirmatory — CIFAR-10
**Stage II:** 800 epochs × 3 seeds, batch 512, LARS, ResNet-18

Proposal Section 3.1: *"Baseline confirmatory runs are trained for 800 epochs with batch size 512
using the LARS optimizer and cosine learning rate decay. We report top-1 accuracy averaged
over three random seeds."*

**Target platform:** PSC Bridges-2 / any A100 (40GB)  
**Estimated runtime:** ~4h/seed × 3 seeds = ~12h total  
**Expected result:** ≥ 90% top-1 linear eval accuracy

## 1. Setup

In [ ]:
import torch, os, subprocess, glob, shutil, csv, datetime
import numpy as np

print(f"PyTorch : {torch.__version__}")
print(f"GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — cannot run confirmatory on CPU'}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")

assert torch.cuda.is_available(), "No GPU detected."

In [ ]:
# ── Configure paths ──────────────────────────────────────────────────────────
REPO_URL   = "https://github.com/SAlaMusa/IDL.git"
REPO_DIR   = os.path.expanduser("~/IDL")          # change if needed on PSC
OUTPUT_DIR = os.path.expanduser("~/outputs/cifar10_confirmatory")  # where to save checkpoints + results
os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

os.chdir(REPO_DIR)
print(f"Repo    : {REPO_DIR}")
print(f"Outputs : {OUTPUT_DIR}")

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)
from models.resnet_simclr import ResNetSimCLR
from optimizers.lars import LARS
print("Imports OK")

## 2. Confirmatory Runs — 3 Seeds
Runs pretraining + linear eval for each seed sequentially.
Results are saved after every seed so a crash mid-run doesn't lose everything.

In [ ]:
SEEDS   = [42, 123, 456]
EPOCHS  = 800
BATCH   = 512    # lr auto-scaled to 0.3 × 512/256 = 0.6
DATASET = "cifar10"
ARCH    = "resnet18"

RESULTS_CSV = os.path.join(OUTPUT_DIR, "cifar10_confirmatory_results.csv")

# Write CSV header once
if not os.path.exists(RESULTS_CSV):
    with open(RESULTS_CSV, "w", newline="") as f:
        csv.writer(f).writerow(["seed", "pretrain_epochs", "batch_size",
                                 "dataset", "arch", "best_top1", "checkpoint", "timestamp"])

completed = {}

for seed in SEEDS:
    ckpt_name = f"{DATASET}_{ARCH}_ep{EPOCHS}_seed{seed}.pth.tar"
    ckpt_path = os.path.join(OUTPUT_DIR, ckpt_name)

    # ── Pretraining ──────────────────────────────────────────────────────────
    if os.path.exists(ckpt_path):
        print(f"[Seed {seed}] Checkpoint already exists, skipping pretraining.")
    else:
        print(f"\n{'='*60}")
        print(f"[Seed {seed}] PRETRAINING — {EPOCHS} epochs, batch {BATCH}")
        print(f"{'='*60}")
        result = subprocess.run([
            "python", "run.py",
            "--config", f"configs/baseline_{DATASET}.yaml",
            "--epochs",     str(EPOCHS),
            "--batch-size", str(BATCH),
            "-j", "4",
            "--seed",       str(seed),
            "--fp16-precision",
        ], capture_output=False)

        if result.returncode != 0:
            raise RuntimeError(f"Pretraining failed for seed {seed}.")

        # Find and save the checkpoint
        checkpoints = glob.glob("runs/**/*.pth.tar", recursive=True)
        latest = max(checkpoints, key=os.path.getmtime)
        shutil.copy2(latest, ckpt_path)
        print(f"[Seed {seed}] Checkpoint saved: {ckpt_path}")

    # ── Linear Evaluation ────────────────────────────────────────────────────
    print(f"\n[Seed {seed}] LINEAR EVALUATION")
    eval_result = subprocess.run([
        "python", "linear_eval.py",
        "--checkpoint", ckpt_path,
        "--dataset",    DATASET,
        "--arch",       ARCH,
        "--epochs",     "100",
        "-b",           "256",
        "-j",           "4",
        "--seed",       str(seed),
    ], capture_output=False)

    if eval_result.returncode != 0:
        raise RuntimeError(f"Linear eval failed for seed {seed}.")

    # Read the best_top1 from the local results CSV written by linear_eval.py
    local_csv = "linear_eval_results.csv"
    with open(local_csv) as f:
        rows = list(csv.DictReader(f))
    best_top1 = float(rows[-1]["best_top1"])
    completed[seed] = best_top1

    # Append to our confirmatory CSV
    with open(RESULTS_CSV, "a", newline="") as f:
        csv.writer(f).writerow([
            seed, EPOCHS, BATCH, DATASET, ARCH,
            f"{best_top1:.2f}", ckpt_path,
            datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
        ])

    print(f"[Seed {seed}] Best Top-1: {best_top1:.2f}%")

print("\nAll seeds complete.")

## 3. Summary — Mean ± Std across Seeds

In [ ]:
accs = list(completed.values())
mean_acc = np.mean(accs)
std_acc  = np.std(accs, ddof=1)   # sample std (ddof=1 for n=3)

print("=" * 50)
print(f"CIFAR-10 SimCLR Baseline — Confirmatory Results")
print(f"Epochs: {EPOCHS}  |  Batch: {BATCH}  |  Arch: {ARCH}")
print("-" * 50)
for seed, acc in completed.items():
    print(f"  Seed {seed}: {acc:.2f}%")
print("-" * 50)
print(f"  Mean ± Std : {mean_acc:.2f} ± {std_acc:.2f}%")
print("=" * 50)
print(f"\nFull results saved to: {RESULTS_CSV}")